# OSUM audio feature extraction (Werewolf-Among-Us, Game3)

Replicates the exact OSUM call used in `MultiMind` (Zhang et al., ACM Multimedia 2025) / its companion code repo `CjangCjengh/onuw` (`start_osum_api.py` + `onuw/mm_utils.py`):
one model call per utterance produces **both** a transcription **and** a categorical vocal-tone label (one of 8: happy, sad, neutral, angry, surprise, disgust, fear, other), via a single Chinese prompt appended-tag trick.

This is the same sample game (`ONE NIGHT ULTIMATE WEREWOLF Retro 3 / Game3`, 35 utterances) already processed locally with Whisper + audEERING's wav2vec2-large-robust (dimensional arousal/valence/dominance) — kept as a separate, independent result for comparison, not a replacement.

**Before running:** Runtime -> Change runtime type -> GPU. OSUM's own README states inference needs **~20GB VRAM** — the free-tier T4 (15GB) may OOM. If it does, this needs Colab Pro (A100/L4) rather than the free tier.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Clone OSUM and install dependencies

In [ ]:
!git clone https://github.com/ASLP-lab/OSUM.git
%cd OSUM/OSUM
# Strip lines we don't want pip to touch:
# - torch_npu: Huawei Ascend hardware only, not applicable on Colab's GPU
# - torch/torchaudio/numpy pins: these are OLD pins (torch==2.1.0, numpy==1.24)
#   that may have no prebuilt wheel for Colab's current Python, forcing a
#   from-source build that fails ("Getting requirements to build wheel").
#   Colab already ships a working torch matched to its CUDA driver - keep it.
# - deepspeed: only needed for distributed training, not inference; its wheel
#   needs a matching CUDA toolkit/nvcc and is a common build-failure culprit.
!sed -i -E '/^(torch_npu|torch|torchaudio|numpy|deepspeed)(==|$| )/d' requirements.txt
!cat requirements.txt
!pip install -r requirements.txt
!pip install huggingface_hub librosa soundfile

## 2. Download the OSUM checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
import os

ckpt_path = hf_hub_download(repo_id="ASLP-lab/OSUM", filename="infer.pt")
os.makedirs("OSUM", exist_ok=True)  # matches the relative path start_osum_api.py expects: 'OSUM/infer.pt'
link_path = "OSUM/infer.pt"
if os.path.lexists(link_path):
    os.remove(link_path)
os.symlink(ckpt_path, link_path)  # symlink instead of copy - instant regardless of checkpoint size
print("Checkpoint ready at", link_path, "->", ckpt_path)

## 3. Rebuild the same sample game's audio (matches the local audEERING run)

Pulls the dialogue annotation JSON from our `werewolf-among-us` branch, downloads the same game video from the HF dataset, extracts audio, and re-creates the identical per-utterance segments.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://raw.githubusercontent.com/praeploykiat/Multimind/werewolf-among-us/raw/train.json -O train.json

import json
GAME_VIDEO_NAME = "ONE NIGHT ULTIMATE WEREWOLF  Retro 3"
GAME_ID = "Game3"
data = json.load(open("train.json", encoding="utf-8"))
game = next(g for g in data if g["video_name"] == GAME_VIDEO_NAME and g["Game_ID"] == GAME_ID)
print(game["video_name"], game["Game_ID"], len(game["Dialogue"]), "utterances")

In [ ]:
from huggingface_hub import hf_hub_download as hf_dl

# Video path within the bolinlai/Werewolf-Among-Us HF dataset (Youtube subset).
# Verified filename: "Youtube/videos/{video_name}_{Game_ID}.mp4" (double space in
# "WEREWOLF  Retro 3" is part of the actual filename, not a typo)
video_path = hf_dl(
    repo_id="bolinlai/Werewolf-Among-Us",
    repo_type="dataset",
    filename=f"Youtube/videos/{game['video_name']}_{game['Game_ID']}.mp4",
)
!ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 game3_audio.wav -loglevel error
print("audio extracted")

In [ ]:
def to_sec(t):
    parts = [int(p) for p in t.split(":")]
    while len(parts) < 3:
        parts.insert(0, 0)
    h, m, s = parts
    return h * 3600 + m * 60 + s

dialogue = game["Dialogue"]
duration = to_sec(game["endTime"]) - to_sec(game["startTime"])
times = [to_sec(d["timestamp"]) for d in dialogue]
windows = []
for i, d in enumerate(dialogue):
    start = times[i]
    end = times[i + 1] if i + 1 < len(dialogue) else duration
    windows.append((start, max(end, start + 0.5)))
print(windows[:5])

## 4. Load OSUM and run the exact prompt used by `onuw`

This mirrors `start_osum_api.py`'s `load_model_on_startup` / `do_decode`, and `onuw/mm_utils.py`'s `_call_transcription_api` prompt + regex parsing, run in-process instead of over HTTP.

In [ ]:
import torch, torchaudio, librosa, numpy as np, re, soundfile as sf
from gxl_ai_utils.utils import utils_file
from wenet.utils.init_tokenizer import init_tokenizer
from gxl_ai_utils.config.gxl_config import GxlNode
from wenet.utils.init_model import init_model

EMOTIONS = ['sad', 'anger', 'neutral', 'happy', 'surprise', 'fear', 'disgust', 'other']
PROMPT = '将音频转录为文字，并在文本最后附加<情感>标签，标签类型涵盖：sad，angry，neutral，happy，surprise，fear，disgust，还有other。'

args = GxlNode({'checkpoint': 'OSUM/infer.pt'})
configs = utils_file.load_dict_from_yaml('examples/osum/conf/config_llm_huawei_base-version.yaml')
model, configs = init_model(args, configs)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
tokenizer = init_tokenizer(configs)
print('OSUM loaded on', device)

In [ ]:
def compute_feat(waveform_1d, sample_rate=16000):
    waveform = torch.from_numpy(waveform_1d).float()
    window = torch.hann_window(400)
    stft = torch.stft(waveform, 400, 160, window=window, return_complex=True)
    magnitudes = stft[..., :-1].abs() ** 2
    filters = torch.from_numpy(librosa.filters.mel(sr=sample_rate, n_fft=400, n_mels=80))
    mel_spec = filters @ magnitudes
    log_spec = torch.clamp(mel_spec, min=1e-10).log10()
    log_spec = torch.maximum(log_spec, log_spec.max() - 8.0)
    log_spec = (log_spec + 4.0) / 4.0
    return log_spec.transpose(0, 1)

def run_osum(wav_np, sample_rate=16000):
    feat = compute_feat(wav_np, sample_rate).unsqueeze(0).to(device)
    feat_lens = torch.tensor([feat.shape[1]], dtype=torch.int64).to(device)
    with torch.no_grad():
        res_text = model.generate(wavs=feat, wavs_len=feat_lens, prompt=PROMPT)[0]
    speech, tone = res_text.strip(), 'other'
    m = re.search(r'^(.*)<(.*)>', res_text.strip())
    if m:
        speech, tone = m.group(1), m.group(2)
    if tone not in EMOTIONS:
        tone = 'other'
    return speech, tone

In [ ]:
full_audio, sr = sf.read('game3_audio.wav', dtype='float32')
assert sr == 16000

results = []
for i, (d, (start, end)) in enumerate(zip(dialogue, windows)):
    seg = full_audio[int(start*sr):int(end*sr)]
    item = {
        'Rec_Id': d['Rec_Id'], 'speaker': d['speaker'], 'timestamp': d['timestamp'],
        'window_sec': [round(start,2), round(end,2)],
        'utterance_ground_truth': d['utterance'], 'annotation': d['annotation'],
    }
    if seg.size < sr * 0.3:
        item['osum_transcript'], item['osum_tone'] = '', 'other'
    else:
        item['osum_transcript'], item['osum_tone'] = run_osum(seg, sr)
    results.append(item)
    print(f"[{i+1}/{len(dialogue)}] {d['speaker']}: tone={item['osum_tone']} text={item['osum_transcript'][:40]!r}")

with open('game3_osum_features.json', 'w', encoding='utf-8') as f:
    json.dump({'game': {'video_name': game['video_name'], 'Game_ID': game['Game_ID'], 'duration_sec': duration},
               'utterances': results}, f, indent=2, ensure_ascii=False)
print('Saved game3_osum_features.json')

## 5. Download the result
Download `game3_osum_features.json` from the Colab file browser (left sidebar) and send it back so it can be merged alongside `game3_audio_features.json` (the audEERING/Whisper result) for comparison.